# OSQP-style solver vs SciPy

This notebook solves an unconstrained convex quadratic program with the local `solvers.osqp` module and compares it against the analytic solution and SciPy's optimizer.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "CMakeLists.txt").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "build"))

import numpy as np
from scipy.optimize import minimize
import osqp

In [4]:
rng = np.random.default_rng(11)
n = 5
M = rng.normal(size=(n, n))
P = M.T @ M + np.eye(n)
q = rng.normal(size=n)

def objective(x):
    return 0.5 * x @ P @ x + q @ x

def gradient(x):
    return P @ x + q

x_star = -np.linalg.solve(P, q)
scipy_result = minimize(objective, np.zeros(n), jac=gradient, method="BFGS", options={"gtol": 1e-10})

settings = osqp.Settings()
settings.enable_ruiz = False
settings.eps_abs = 1e-6
settings.eps_rel = 1e-6
A = np.zeros((0, n))
l = np.zeros(0)
u = np.zeros(0)
osqp_result = osqp.solve(P, q, A, l, u, settings)
x_osqp = osqp_result["x"]

print("OSQP status:", osqp_result["status"])
print("OSQP x:", x_osqp)
print("Analytic x:", x_star)
print("Analytic max error:", np.max(np.abs(x_osqp - x_star)))
print("SciPy max error:", np.max(np.abs(x_osqp - scipy_result.x)))
print("Objective values:", objective(x_osqp), objective(x_star), scipy_result.fun)

OSQP status: solved
OSQP x: [ 0.05145893 -0.20496615  0.82260608  1.05828845  0.16137147]
Analytic x: [ 0.05145892 -0.20496613  0.82260597  1.05828831  0.16137145]
Analytic max error: 1.403660221388492e-07
SciPy max error: 1.4036602169475998e-07
Objective values: -1.2505644695138713 -1.2505644695138936 -1.2505644695138936


In [3]:
assert osqp_result["status"] == "solved"
assert scipy_result.success
assert np.allclose(x_osqp, x_star, rtol=1e-3, atol=1e-4)